In [ ]:
import sys

PROJECT_ROOT = "/content/drive/MyDrive/green-ai-cnn-carbon"
sys.path.append(PROJECT_ROOT)

print("Project root added")

In [ ]:
!pip install -q codecarbon

In [ ]:
import tensorflow as tf
import numpy as np
import pandas as pd
import time
import os

from tensorflow.keras.datasets import cifar100
from tensorflow.keras.utils import to_categorical
from codecarbon import EmissionsTracker

In [ ]:
BASE_RESULTS_DIR = "/content/drive/MyDrive/green-ai-cnn-carbon/results/cifar100"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

In [ ]:
(x_train, y_train), (x_test, y_test) = cifar100.load_data(label_mode="fine")

x_train = x_train.astype("float32") / 255.0
x_test  = x_test.astype("float32") / 255.0

y_train = to_categorical(y_train, 100)
y_test  = to_categorical(y_test, 100)

In [ ]:
def get_lenet():
    model = tf.keras.Sequential([
        tf.keras.layers.Conv2D(6, (5,5), activation='relu', input_shape=(32,32,3)),
        tf.keras.layers.AveragePooling2D(pool_size=(2,2)),

        tf.keras.layers.Conv2D(16, (5,5), activation='relu'),
        tf.keras.layers.AveragePooling2D(pool_size=(2,2)),

        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(120, activation='relu'),
        tf.keras.layers.Dense(84, activation='relu'),
        tf.keras.layers.Dense(100, activation='softmax')  # CIFAR-100
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

    return model


In [ ]:
EPOCHS = 50
BATCH_SIZE = 64
MODEL_NAME = "lenet"

# Build model
model = get_lenet()

# Use correct input variables
x_train_model = x_train
x_test_model  = x_test

# CodeCarbon tracker
tracker = EmissionsTracker(
    project_name=f"carbon_budget_{MODEL_NAME}_cifar100",
    output_dir="/content/drive/MyDrive/green-ai-cnn-carbon/tracking",
    log_level="error"
)

epoch_logs = []

tracker.start()

for epoch in range(EPOCHS):
    model.fit(
        x_train_model, y_train,
        batch_size=BATCH_SIZE,
        epochs=1,
        verbose=1
    )

    loss, acc = model.evaluate(
        x_test_model, y_test,
        verbose=0
    )

    emissions_g = tracker.flush() * 1000  # kg → grams

    epoch_logs.append({
        "dataset": "CIFAR-100",
        "model": MODEL_NAME,
        "epoch": epoch + 1,
        "val_accuracy": acc,
        "cumulative_co2_g": emissions_g
    })

    print(
        f"Epoch {epoch+1:02d} | "
        f"Val Acc: {acc:.4f} | "
        f"CO₂: {emissions_g:.2f} g"
    )

tracker.stop()
df = pd.DataFrame(epoch_logs)
csv_path = f"{BASE_RESULTS_DIR}/epochwise_carbon_{MODEL_NAME}.csv"
df.to_csv(csv_path, index=False)

print("Saved:", csv_path)


In [ ]:
df = pd.read_csv(CSV_PATH)

df[
    (df["dataset"] == "cifar100") &
    (df["training_mode"] == "convergence") &
    (df["model"] == MODEL_NAME)
].tail(3)